In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [3]:
x = torch.rand((4,6))
linear = nn.Linear(6, 10)
x

tensor([[0.0398, 0.2377, 0.2936, 0.2662, 0.6775, 0.4081],
        [0.3967, 0.3441, 0.0368, 0.0639, 0.4954, 0.2169],
        [0.6972, 0.7980, 0.1181, 0.9105, 0.3668, 0.7730],
        [0.2559, 0.7096, 0.0519, 0.3352, 0.5222, 0.1635]])

In [31]:
# GLOBALS 
token_amount = 50
B,T,C  = 4, 8 ,32 # Batch size, time steps (seq length), channels (embedding vector size per token)


In [4]:
x = linear(x)
x

tensor([[-0.3215, -0.0978,  0.5449,  0.1105, -0.3168, -0.0736, -0.2930,  0.1824,
         -0.0447, -0.0983],
        [-0.2821, -0.1116,  0.5392,  0.1572, -0.3935,  0.0612, -0.1843,  0.3002,
         -0.1419, -0.2104],
        [-0.4147,  0.0654,  0.7106,  0.5663, -0.4817,  0.5479, -0.0380,  0.5991,
          0.1216,  0.0250],
        [-0.3470, -0.0144,  0.7215,  0.2863, -0.2545,  0.1750, -0.2397,  0.4472,
          0.1571, -0.0247]], grad_fn=<AddmmBackward0>)

In [ ]:
ten = 2 * torch.rand((5,5)) - 1
ten

tensor([[-0.1999,  0.3500, -0.9665,  0.1879,  0.9494],
        [-0.5176, -0.2678,  0.5901,  0.5909, -0.8345],
        [-0.9984,  0.4059, -0.6215,  0.6637, -0.9719],
        [-0.8647, -0.9612,  0.0168,  0.5934, -0.1301],
        [-0.8543,  0.7611,  0.4418,  0.6282, -0.0487]])

In [ ]:
approx = 'None' # 'tanh'
gelu = nn.GELU()
y = gelu(ten)
y

tensor([[-0.0841,  0.2229, -0.1613,  0.1080,  0.7869],
        [-0.1565, -0.1056,  0.4263,  0.4271, -0.1686],
        [-0.1588,  0.2669, -0.1660,  0.4955, -0.1609],
        [-0.1674, -0.1617,  0.0085,  0.4294, -0.0583],
        [-0.1678,  0.5911,  0.2963,  0.4618, -0.0234]])

In [ ]:
relu = nn.ReLU()
y_rel = relu(ten)
y_rel

tensor([[0.0000, 0.3500, 0.0000, 0.1879, 0.9494],
        [0.0000, 0.0000, 0.5901, 0.5909, 0.0000],
        [0.0000, 0.4059, 0.0000, 0.6637, 0.0000],
        [0.0000, 0.0000, 0.0168, 0.5934, 0.0000],
        [0.0000, 0.7611, 0.4418, 0.6282, 0.0000]])

In [ ]:
emb = nn.Embedding()

In [56]:
torch.manual_seed(47)
torch.rand((3,6))

tensor([[0.0530, 0.0499, 0.4677, 0.8757, 0.5561, 0.7984],
        [0.9758, 0.2482, 0.1469, 0.4345, 0.6988, 0.8883],
        [0.2638, 0.2658, 0.1375, 0.4610, 0.7439, 0.0351]])

In [9]:
B,T,C  = 4, 8 ,32
x = torch.rand(B,T,C)

In [10]:
x.shape

torch.Size([4, 8, 32])

In [11]:
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

In [12]:
k = key(x)
q = query(x)
k.shape, q.shape

(torch.Size([4, 8, 16]), torch.Size([4, 8, 16]))

In [13]:
wei = q @ k.transpose(-1, -2)
tril = torch.tril(torch.ones(T, T))
tril

tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])

In [14]:
wei  = wei.masked_fill(tril == 0, float('-inf'))
wei

tensor([[[-4.8895e-01,        -inf,        -inf,        -inf,        -inf,
                 -inf,        -inf,        -inf],
         [-2.3452e-01,  2.1036e-02,        -inf,        -inf,        -inf,
                 -inf,        -inf,        -inf],
         [-3.4460e-01, -2.7755e-01, -3.1090e-01,        -inf,        -inf,
                 -inf,        -inf,        -inf],
         [-5.4829e-01, -3.8064e-01, -4.9579e-01, -7.1533e-01,        -inf,
                 -inf,        -inf,        -inf],
         [-4.4412e-01, -3.3174e-01, -7.5858e-01, -7.9724e-01, -3.9747e-01,
                 -inf,        -inf,        -inf],
         [-1.2590e-01,  1.7571e-01,  6.7290e-02,  2.7561e-01, -4.5026e-02,
           1.0393e-01,        -inf,        -inf],
         [-6.0191e-01, -5.6386e-01, -4.6082e-01, -7.8845e-01, -3.7499e-01,
          -4.9949e-01, -2.5839e-01,        -inf],
         [-6.9999e-01, -6.7810e-01, -1.1653e+00, -1.2604e+00, -6.1930e-01,
          -1.1798e+00, -4.7114e-01, -1.1711e+00]],

In [15]:
wei = F.softmax(wei, dim=1)
wei

tensor([[[0.1166, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1504, 0.1874, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1347, 0.1390, 0.1917, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1099, 0.1254, 0.1593, 0.1633, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1220, 0.1317, 0.1225, 0.1504, 0.2355, 0.0000, 0.0000, 0.0000],
         [0.1677, 0.2188, 0.2798, 0.4398, 0.3350, 0.5483, 0.0000, 0.0000],
         [0.1042, 0.1044, 0.1650, 0.1518, 0.2409, 0.2999, 0.5530, 0.0000],
         [0.0944, 0.0932, 0.0816, 0.0947, 0.1886, 0.1519, 0.4470, 1.0000]],

        [[0.1144, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1146, 0.1476, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1581, 0.1809, 0.2407, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1180, 0.1587, 0.1911, 0.2208, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1692, 0.1591, 0.1766, 0.2656, 0.2884, 0.0000, 0.0000, 0.0000],
         [0.0807, 0.085

In [16]:
wei.shape

torch.Size([4, 8, 8])

In [17]:
v = value(x)

In [18]:
v.shape

torch.Size([4, 8, 16])

In [19]:
att = wei @ v

In [20]:
att.shape

torch.Size([4, 8, 16])

In [23]:
test = torch.ones(5)

In [27]:
test[0:2] = test[:2] + 3

In [30]:
test.std()

tensor(1.6432)

In [34]:
x.shape

torch.Size([4, 8, 32])

In [32]:
emb = nn.Embedding(token_amount, C)

In [71]:
input_sequences = torch.randint(0, token_amount, (B, T))
input_sequences

tensor([[20, 28, 18,  0, 13, 10, 46, 41],
        [26,  7, 16, 41, 21, 38, 32,  1],
        [41, 45, 22, 20,  5,  0, 17, 24],
        [40, 43, 30,  2,  1, 47, 33, 45]])

In [72]:
# Here we're doing the indexing. For each sequence example, fetch the corresponding embedding matrix.
# `input_sequences` here should be B examples of T token sequences in which each T element is a token ID not some rand value
input_embedding = emb(input_sequences)
input_embedding.shape # B T C

torch.Size([4, 8, 32])